# Leak-free Modeling Pipeline (Spotify)
This notebook rebuilds the modeling workflow using an sklearn `Pipeline` + `ColumnTransformer` so that **all preprocessing is fit on the training set only**.

Key differences vs a typical quick notebook:
- Split first (train/val/test)
- Fit scaler/encoder on train only
- Avoid undersampling unless you explicitly want it (we use `class_weight='balanced'`)

Target used here:
- `is_hit = 1` if `popularity > 50` (raw popularity is 0–100)

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)

In [9]:
df_raw = pd.read_csv('Spotify_Music.csv')
print('shape:', df_raw.shape)
df_raw.head()

shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [10]:
# Drop high-cardinality text identifiers (they're not audio features)
drop_cols = ['track_id', 'artists', 'album_name', 'track_name', 'Unnamed: 0']
df = df_raw.drop(columns=drop_cols, errors='ignore').copy()

# Define target from raw popularity (0..100)
if 'popularity' not in df.columns:
    raise ValueError('Expected a popularity column in the dataset.')

df['is_hit'] = (df['popularity'] > 50).astype(int)

# Features: exclude popularity/label and (optionally) genre for an audio-only baseline
X = df.drop(columns=['popularity', 'is_hit', 'track_genre'], errors='ignore')
y = df['is_hit']

print('X shape:', X.shape)
print('y mean (hit rate):', y.mean())
y.value_counts().sort_index()

X shape: (114000, 14)
y mean (hit rate): 0.24359649122807017


is_hit
0    86230
1    27770
Name: count, dtype: int64

In [11]:
# 70% train, 15% val, 15% test (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('train:', X_train.shape, 'hit rate:', y_train.mean())
print('val:  ', X_val.shape, 'hit rate:', y_val.mean())
print('test: ', X_test.shape, 'hit rate:', y_test.mean())

train: (79800, 14) hit rate: 0.24359649122807017
val:   (17100, 14) hit rate: 0.24362573099415205
test:  (17100, 14) hit rate: 0.2435672514619883


In [12]:
# Auto-detect columns for preprocessing
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

print('categorical:', cat_cols)
print('numeric:', len(num_cols))

numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, num_cols),
        ('cat', categorical_pipe, cat_cols),
    ],
    remainder='drop',
)

categorical: []
numeric: 14


In [13]:
# Models (no undersampling; use class_weight to address imbalance)
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    'Linear SVM (calibrated)': CalibratedClassifierCV(
        estimator=LinearSVC(random_state=42, max_iter=20000, class_weight='balanced'),
        method='sigmoid',
        cv=3,
    ),
}

def evaluate(name, pipe, X_tr, y_tr, X_va, y_va, X_te, y_te):
    pipe.fit(X_tr, y_tr)

    def _pred_score(p, X_):
        y_pred_ = p.predict(X_)
        y_score_ = None
        y_proba2_ = None
        if hasattr(p, 'predict_proba'):
            proba = p.predict_proba(X_)
            # binary proba -> 2 columns
            if proba.ndim == 2 and proba.shape[1] == 2:
                y_proba2_ = proba
                y_score_ = proba[:, 1]
        if y_score_ is None and hasattr(p, 'decision_function'):
            y_score_ = p.decision_function(X_)
        return y_pred_, y_score_, y_proba2_

    y_tr_pred, y_tr_score, y_tr_proba2 = _pred_score(pipe, X_tr)
    y_va_pred, y_va_score, y_va_proba2 = _pred_score(pipe, X_va)
    y_te_pred, y_te_score, y_te_proba2 = _pred_score(pipe, X_te)

    def _metrics(y_true, y_pred, y_score, y_proba2):
        out = {
            'Acc': accuracy_score(y_true, y_pred),
            'Bal Acc': balanced_accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred),
            'ROC AUC': roc_auc_score(y_true, y_score) if y_score is not None else float('nan'),
            'PR AUC': average_precision_score(y_true, y_score) if y_score is not None else float('nan'),
            'Log Loss': log_loss(y_true, y_proba2) if y_proba2 is not None else float('nan'),
        }
        return out

    tr_m = _metrics(y_tr, y_tr_pred, y_tr_score, y_tr_proba2)
    va_m = _metrics(y_va, y_va_pred, y_va_score, y_va_proba2)
    te_m = _metrics(y_te, y_te_pred, y_te_score, y_te_proba2)

    print(f'\n==== {name} ====')
    print('Validation report:')
    print(classification_report(y_va, y_va_pred, digits=4))
    print('Confusion matrix (val):')
    print(confusion_matrix(y_va, y_va_pred))

    row = {
        'Model': name,
        'Train Acc': tr_m['Acc'], 'Val Acc': va_m['Acc'], 'Test Acc': te_m['Acc'],
        'Train Bal Acc': tr_m['Bal Acc'], 'Val Bal Acc': va_m['Bal Acc'], 'Test Bal Acc': te_m['Bal Acc'],
        'Train Precision': tr_m['Precision'], 'Val Precision': va_m['Precision'], 'Test Precision': te_m['Precision'],
        'Train Recall': tr_m['Recall'], 'Val Recall': va_m['Recall'], 'Test Recall': te_m['Recall'],
        'Train F1': tr_m['F1'], 'Val F1': va_m['F1'], 'Test F1': te_m['F1'],
        'Train ROC AUC': tr_m['ROC AUC'], 'Val ROC AUC': va_m['ROC AUC'], 'Test ROC AUC': te_m['ROC AUC'],
        'Train PR AUC': tr_m['PR AUC'], 'Val PR AUC': va_m['PR AUC'], 'Test PR AUC': te_m['PR AUC'],
        'Train Loss': tr_m['Log Loss'], 'Val Loss': va_m['Log Loss'], 'Test Loss': te_m['Log Loss'],
    }
    return row

rows = []
for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', clf)])
    rows.append(evaluate(name, pipe, X_train, y_train, X_val, y_val, X_test, y_test))

results_df = pd.DataFrame(rows)
results_df.sort_values('Val F1', ascending=False)


==== Logistic Regression ====
Validation report:
              precision    recall  f1-score   support

           0     0.8193    0.5441    0.6539     12934
           1     0.3071    0.6275    0.4124      4166

    accuracy                         0.5644     17100
   macro avg     0.5632    0.5858    0.5332     17100
weighted avg     0.6945    0.5644    0.5951     17100

Confusion matrix (val):
[[7037 5897]
 [1552 2614]]

==== Random Forest ====
Validation report:
              precision    recall  f1-score   support

           0     0.8429    0.9784    0.9056     12934
           1     0.8663    0.4340    0.5783      4166

    accuracy                         0.8458     17100
   macro avg     0.8546    0.7062    0.7420     17100
weighted avg     0.8486    0.8458    0.8259     17100

Confusion matrix (val):
[[12655   279]
 [ 2358  1808]]

==== Linear SVM (calibrated) ====
Validation report:
              precision    recall  f1-score   support

           0     0.7570    0.9970    

,Model,Train Acc,Val Acc,Test Acc,Train Bal Acc,Val Bal Acc,Test Bal Acc,Train Precision,Val Precision,Test Precision,...,Test F1,Train ROC AUC,Val ROC AUC,Test ROC AUC,Train PR AUC,Val PR AUC,Test PR AUC,Train Loss,Val Loss,Test Loss
1,Random Forest,0.992105,0.845789,0.848012,0.991119,0.706209,0.708669,0.978625,0.866315,0.877531,...,0.583427,0.998959,0.854091,0.854462,0.996283,0.724694,0.731295,0.111441,0.403247,0.391168
0,Logistic Regression,0.567607,0.564386,0.564386,0.587787,0.585765,0.586389,0.309040,0.307132,0.307413,...,0.413049,0.622972,0.622590,0.622532,0.332562,0.328775,0.328335,0.668414,0.668798,0.668074
2,Linear SVM (calibrated),0.755852,0.755614,0.755731,0.501536,0.501613,0.501245,0.416031,0.400000,0.388889,...,0.009955,0.622884,0.622725,0.622199,0.332219,0.328547,0.327567,0.536843,0.536889,0.536674


In [14]:
# Threshold tuning for imbalanced classification
from sklearn.base import clone
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    log_loss,
    balanced_accuracy_score,
    accuracy_score,
 )

# Pick the model using a threshold-independent metric (PR AUC works well for imbalance)
best_name = results_df.sort_values('Val PR AUC', ascending=False).iloc[0]['Model']
print('Selected model (by Val PR AUC):', best_name)

best_pipe = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', clone(models[best_name])),
 ])
best_pipe.fit(X_train, y_train)

# Scores on validation / test
val_proba = best_pipe.predict_proba(X_val)[:, 1]
test_proba = best_pipe.predict_proba(X_test)[:, 1]

print('Val ROC AUC:', roc_auc_score(y_val, val_proba))
print('Val PR  AUC:', average_precision_score(y_val, val_proba))

# Sweep thresholds and pick the best one for F1 (change to F2 if you care more about recall)
thresholds = np.linspace(0.05, 0.95, 91)
rows_thr = []
for t in thresholds:
    val_pred_t = (val_proba >= t).astype(int)
    rows_thr.append({
        'threshold': float(t),
        'val_acc': accuracy_score(y_val, val_pred_t),
        'val_bal_acc': balanced_accuracy_score(y_val, val_pred_t),
        'val_precision': precision_score(y_val, val_pred_t, zero_division=0),
        'val_recall': recall_score(y_val, val_pred_t, zero_division=0),
        'val_f1': f1_score(y_val, val_pred_t),
    })

thr_df = pd.DataFrame(rows_thr)
best_t = thr_df.sort_values('val_f1', ascending=False).iloc[0]['threshold']
print('\nBest threshold on VAL (by F1):', best_t)
display(thr_df.sort_values('val_f1', ascending=False).head(10))

# Compare 0.5 vs tuned threshold on TEST
def report_at_threshold(split_name, y_true, proba, t):
    pred = (proba >= t).astype(int)
    print(f'\n--- {split_name} @ threshold={t:.2f} ---')
    print(classification_report(y_true, pred, digits=4))
    print('confusion matrix:')
    print(confusion_matrix(y_true, pred))
    return pred

_ = report_at_threshold('VAL', y_val, val_proba, 0.50)
_ = report_at_threshold('VAL', y_val, val_proba, float(best_t))
_ = report_at_threshold('TEST', y_test, test_proba, 0.50)
_ = report_at_threshold('TEST', y_test, test_proba, float(best_t))

Selected model (by Val PR AUC): Random Forest
Val ROC AUC: 0.8540911961098561
Val PR  AUC: 0.7246935798600449

Best threshold on VAL (by F1): 0.35999999999999993


,threshold,val_acc,val_bal_acc,val_precision,val_recall,val_f1
31,0.36,0.828129,0.742294,0.672186,0.574892,0.619744
32,0.37,0.831345,0.740189,0.688308,0.562410,0.619022
27,0.32,0.808596,0.750861,0.600904,0.638262,0.619020
26,0.31,0.802339,0.752583,0.584046,0.655545,0.617734
29,0.34,0.818070,0.745245,0.632838,0.603217,0.617672
28,0.33,0.812865,0.747256,0.615165,0.619299,0.617225
30,0.35,0.821404,0.742323,0.646779,0.588094,0.616042
25,0.30,0.793977,0.753238,0.564675,0.673788,0.614425
33,0.38,0.832807,0.735379,0.701884,0.545367,0.613805
24,0.29,0.787135,0.755550,0.550038,0.693951,0.613670



--- VAL @ threshold=0.50 ---
              precision    recall  f1-score   support

           0     0.8434    0.9780    0.9057     12934
           1     0.8645    0.4364    0.5800      4166

    accuracy                         0.8460     17100
   macro avg     0.8540    0.7072    0.7429     17100
weighted avg     0.8486    0.8460    0.8264     17100

confusion matrix:
[[12649   285]
 [ 2348  1818]]

--- VAL @ threshold=0.36 ---
              precision    recall  f1-score   support

           0     0.8692    0.9097    0.8890     12934
           1     0.6722    0.5749    0.6197      4166

    accuracy                         0.8281     17100
   macro avg     0.7707    0.7423    0.7544     17100
weighted avg     0.8212    0.8281    0.8234     17100

confusion matrix:
[[11766  1168]
 [ 1771  2395]]

--- TEST @ threshold=0.50 ---
              precision    recall  f1-score   support

           0     0.8443    0.9800    0.9071     12935
           1     0.8758    0.4387    0.5845     

## Precision-Focused Grid Search (More Features)
This section runs a more detailed `GridSearchCV` using a **richer feature set** (includes `track_genre` via one-hot encoding) and selects hyperparameters using **precision** (with a minimum-recall guard so it doesn’t “cheat” by predicting almost no hits).

> Note: Grid search is run **only on the training split** using stratified CV. Validation and test remain held out for final evaluation.

In [15]:
# Build a richer feature set by keeping track_genre (one-hot encoded)
df2 = df_raw.drop(columns=drop_cols, errors='ignore').copy()
df2['is_hit'] = (df2['popularity'] > 50).astype(int)

X2 = df2.drop(columns=['popularity', 'is_hit'], errors='ignore')  # includes track_genre
y2 = df2['is_hit']

print('X2 shape:', X2.shape)
print('categorical cols:', X2.select_dtypes(include=['object', 'category']).columns.tolist())
print('hit rate:', y2.mean())

X2 shape: (114000, 15)
categorical cols: ['track_genre']
hit rate: 0.24359649122807017


In [16]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer

# Split (same ratios)
X2_train, X2_temp, y2_train, y2_temp = train_test_split(
    X2, y2, test_size=0.30, random_state=42, stratify=y2
)
X2_val, X2_test, y2_val, y2_test = train_test_split(
    X2_temp, y2_temp, test_size=0.50, random_state=42, stratify=y2_temp
)

print('train:', X2_train.shape, 'hit rate:', y2_train.mean())
print('val:  ', X2_val.shape, 'hit rate:', y2_val.mean())
print('test: ', X2_test.shape, 'hit rate:', y2_test.mean())

# Preprocess (make OHE dense to keep tree models happy)
cat2 = X2_train.select_dtypes(include=['object', 'category']).columns.tolist()
num2 = [c for c in X2_train.columns if c not in cat2]

numeric_pipe2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler()),
])
categorical_pipe2 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocess2 = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe2, num2),
        ('cat', categorical_pipe2, cat2),
    ],
    remainder='drop',
 )

print('num2:', len(num2), 'cat2:', cat2)

train: (79800, 15) hit rate: 0.24359649122807017
val:   (17100, 15) hit rate: 0.24362573099415205
test:  (17100, 15) hit rate: 0.2435672514619883
num2: 14 cat2: ['track_genre']


In [17]:
# Precision-focused scoring with a minimum recall constraint
MIN_RECALL = 0.40  # change this based on how many hits you MUST catch
print('MIN_RECALL constraint:', MIN_RECALL)

def precision_with_min_recall(y_true, y_pred, min_recall=MIN_RECALL):
    r = recall_score(y_true, y_pred, zero_division=0)
    if r < min_recall:
        return 0.0
    return precision_score(y_true, y_pred, zero_division=0)

scoring = {
    'prec': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score),
    'ap': 'average_precision',
    'roc_auc': 'roc_auc',
    'prec_minR': make_scorer(precision_with_min_recall, min_recall=MIN_RECALL),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

MIN_RECALL constraint: 0.4


In [18]:
# Define model grids (kept moderate so it finishes in a reasonable time)
search_spaces = []

# Logistic Regression grid
lr = LogisticRegression(max_iter=5000)
lr_grid = {
    'model__C': [0.05, 0.1, 0.5, 1.0, 5.0],
    'model__solver': ['liblinear', 'saga'],
    'model__class_weight': [None, 'balanced'],
}
search_spaces.append(('LogReg', lr, lr_grid))

# Random Forest grid
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid = {
    'model__n_estimators': [300, 600],
    'model__max_depth': [None, 20, 40],
    'model__min_samples_leaf': [1, 4],
    'model__max_features': ['sqrt', 0.5],
    'model__class_weight': [None, 'balanced', 'balanced_subsample'],
}
search_spaces.append(('RF', rf, rf_grid))

# LightGBM grid (if available)
try:
    from lightgbm import LGBMClassifier
    pos = int((y2_train == 1).sum())
    neg = int((y2_train == 0).sum())
    scale_pos_weight = (neg / pos) if pos else 1.0
    lgbm = LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        objective='binary',
        scale_pos_weight=scale_pos_weight,
    )
    lgbm_grid = {
        'model__n_estimators': [300, 800],
        'model__learning_rate': [0.05, 0.1],
        'model__num_leaves': [31, 63],
        'model__max_depth': [-1, 10],
        'model__subsample': [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0],
    }
    search_spaces.append(('LGBM', lgbm, lgbm_grid))
    print('LightGBM enabled. scale_pos_weight=', scale_pos_weight)
except Exception as e:
    print('LightGBM not available, skipping. Reason:', repr(e))

len(search_spaces)

LightGBM enabled. scale_pos_weight= 3.105149441843716


3

In [ ]:
# Run grid searches (refit = precision with min recall)
grid_results = []
best_estimators = {}

for tag, base_model, param_grid in search_spaces:
    print(f'\n================ Grid Search: {tag} ================')
    pipe = Pipeline(steps=[('preprocess', preprocess2), ('model', base_model)])
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=scoring,
        refit='prec_minR',
        cv=cv,
        n_jobs=-1,
        verbose=2,
        return_train_score=False,
    )
    gs.fit(X2_train, y2_train)

    best_estimators[tag] = gs.best_estimator_
    best = {
        'Tag': tag,
        'Best Params': gs.best_params_,
        'CV prec_minR': gs.best_score_,
        'CV precision': gs.cv_results_['mean_test_prec'][gs.best_index_],
        'CV recall': gs.cv_results_['mean_test_recall'][gs.best_index_],
        'CV f1': gs.cv_results_['mean_test_f1'][gs.best_index_],
        'CV AP': gs.cv_results_['mean_test_ap'][gs.best_index_],
        'CV ROC AUC': gs.cv_results_['mean_test_roc_auc'][gs.best_index_],
    }
    grid_results.append(best)
    print('Best:', best)

grid_results_df = pd.DataFrame(grid_results)
grid_results_df.sort_values('CV prec_minR', ascending=False)


================ Grid Search: LogReg ================
Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
# Pick best model from grid search and tune threshold on VAL for precision (with min recall)
best_tag = grid_results_df.sort_values('CV prec_minR', ascending=False).iloc[0]['Tag']
best_est = best_estimators[best_tag]
print('Best grid-search model:', best_tag)

best_est.fit(X2_train, y2_train)
val_proba2 = best_est.predict_proba(X2_val)[:, 1]
test_proba2 = best_est.predict_proba(X2_test)[:, 1]

# threshold sweep on VAL: maximize precision subject to recall >= MIN_RECALL
thresholds = np.linspace(0.05, 0.95, 91)
rows_thr2 = []
for t in thresholds:
    val_pred = (val_proba2 >= t).astype(int)
    p = precision_score(y2_val, val_pred, zero_division=0)
    r = recall_score(y2_val, val_pred, zero_division=0)
    f1v = f1_score(y2_val, val_pred)
    if r >= MIN_RECALL:
        rows_thr2.append({'threshold': float(t), 'precision': p, 'recall': r, 'f1': f1v})

thr2 = pd.DataFrame(rows_thr2)
if thr2.empty:
    print('No threshold met MIN_RECALL on validation. Lower MIN_RECALL and rerun.')
else:
    best_t2 = thr2.sort_values(['precision', 'recall'], ascending=False).iloc[0]['threshold']
    print('Best threshold on VAL (maximize precision, recall>=MIN_RECALL):', best_t2)
    display(thr2.sort_values(['precision','recall'], ascending=False).head(10))

    # Final test report
    test_pred_05 = (test_proba2 >= 0.50).astype(int)
    test_pred_t = (test_proba2 >= best_t2).astype(int)

    print('\nTEST @ 0.50')
    print(classification_report(y2_test, test_pred_05, digits=4))
    print('confusion matrix:')
    print(confusion_matrix(y2_test, test_pred_05))

    print(f'\nTEST @ tuned threshold={best_t2:.2f}')
    print(classification_report(y2_test, test_pred_t, digits=4))
    print('confusion matrix:')
    print(confusion_matrix(y2_test, test_pred_t))

## Notes
If your scores drop a bit compared to the earlier notebook, that's often a sign the earlier workflow had **data leakage** from scaling/encoding before splitting.

If you want, we can also:
- Change the hit threshold (e.g., top 20% instead of >50)
- Try regression on popularity instead of classification
- Add calibration / threshold tuning based on validation PR curve